In [19]:
import cv2
import numpy as np
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

import re

# --- STEP 1: Read the image ---
img = cv2.imread(r'C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg')
if img is None:
    raise SystemExit("Image not found! Check your file path.")

# --- STEP 2: Convert to grayscale ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# --- STEP 3: Enhance contrast and reduce noise ---
# Optional: increase brightness/contrast
gray = cv2.convertScaleAbs(gray, alpha=1.5, beta=10)

# Blur to remove small noise
blur = cv2.GaussianBlur(gray, (3, 3), 0)

# --- STEP 4: Threshold (binarize) ---
# Use Otsu’s automatic thresholding
_, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# If digits appear white on black, invert it
# (You can comment this out if inversion makes it worse)
thresh = cv2.bitwise_not(thresh)

# --- STEP 5: (Optional) Crop region of interest ---
# Adjust this if you know where digits are on the screen
# Example: crop middle of image
h, w = thresh.shape
roi = thresh[int(h * 0.25):int(h * 0.75), int(w * 0.1):int(w * 0.9)]

# --- STEP 6: OCR - extract only digits ---
custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789.'
ocr_text = pytesseract.image_to_string(roi, config=custom_config)
ocr_text = ocr_text.strip()

print("🔹 Raw OCR Text:", repr(ocr_text))

# --- STEP 7: Extract all numeric values using regex ---
numbers = re.findall(r"\d+\.\d+|\d+", ocr_text)
print("🔹 Extracted Digits:", numbers)

# --- STEP 8: Convert to numbers (int/float) ---
digits_cleaned = [float(n) if '.' in n else int(n) for n in numbers]
print("✅ Cleaned Digits List:", digits_cleaned)

# --- STEP 9: Show preprocessing result ---
cv2.imshow("Grayscale", gray)
cv2.imshow("Thresholded", thresh)
cv2.imshow("ROI for OCR", roi)
cv2.waitKey(0)
cv2.destroyAllWindows()


🔹 Raw OCR Text: '2027\n2'
🔹 Extracted Digits: ['2027', '2']
✅ Cleaned Digits List: [2027, 2]


In [6]:

import cv2
import numpy as np
import pytesseract
import re

# Path to Tesseract (adjust if installed elsewhere)
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# --- STEP 1: Read the image ---
img = cv2.imread(r'C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg')
if img is None:
    raise SystemExit("❌ Image not found! Check your file path.")

# --- STEP 2: Convert to Grayscale ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# --- STEP 3: OPTIONAL - Improve visibility (contrast and brightness) ---
gray = cv2.convertScaleAbs(gray, alpha=1.5, beta=10)

# --- STEP 4: Manually Crop the Region where digits appear ---
# You can adjust these values based on where your digits appear in the image
h, w = gray.shape
# Example: top 30% to 50% vertically, left 10% to 50% horizontally
cropped = gray[int(h * 0.3):int(h * 0.5), int(w * 0.1):int(w * 0.5)]

# --- STEP 5: Preprocess the cropped area for OCR ---
blur = cv2.GaussianBlur(cropped, (3, 3), 0)
_, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# If digits are white on black, invert
thresh = cv2.bitwise_not(thresh)

# --- STEP 6: OCR for only digits ---
custom_config = r'--oem 3 --psm 7 -c tessedit_char_whitelist=0123456789.'
ocr_text = pytesseract.image_to_string(thresh, config=custom_config).strip()

print("🔹 Raw OCR Text:", repr(ocr_text))

# --- STEP 7: Extract numeric digits ---
numbers = re.findall(r"\d+\.\d+|\d+", ocr_text)
print("🔹 Extracted Digits:", numbers)

# --- STEP 8: Clean them into integers/floats ---
if numbers:
    digits_cleaned = [float(n) if '.' in n else int(n) for n in numbers]
    print("✅ Cleaned Digits List:", digits_cleaned)
else:
    print("⚠️ No digits detected.")

# --- STEP 9: Visualize results ---
cv2.imshow("Original Grayscale", gray)
cv2.imshow("Cropped Region", cropped)
cv2.imshow("Processed (Thresholded)", thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


🔹 Raw OCR Text: ''
🔹 Extracted Digits: []
⚠️ No digits detected.


In [28]:
import cv2
import numpy as np
import pytesseract
import re

# --- STEP 1: Path to Tesseract (adjust if installed elsewhere) ---
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# --- STEP 2: Read the image ---
img_path = r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg"
img = cv2.imread(img_path)
if img is None:
    raise SystemExit("❌ Image not found! Check the file path.")

# --- STEP 3: Convert to Grayscale ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# --- STEP 4: Enhance contrast and brightness ---
gray = cv2.convertScaleAbs(gray, alpha=1.5, beta=30)  # alpha=contrast, beta=brightness

# --- STEP 5: Manually Crop Region of Interest (ROI) ---
# Adjust these percentages depending on where the digits appear in your image
h, w = gray.shape
roi = gray[int(h * 0.3):int(h * 0.5), int(w * 0.1):int(w * 0.5)]

# --- STEP 6: Preprocess (remove noise, binarize, and invert) ---
blur = cv2.GaussianBlur(roi, (3, 3), 0)

# Adaptive threshold is more robust than Otsu for varying lighting
thresh = cv2.adaptiveThreshold(
    blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
)

# Invert so digits become black on white (better for Tesseract)
thresh = cv2.bitwise_not(thresh)

# Optional: Make digits thicker for clarity
kernel = np.ones((2, 2), np.uint8)
thresh = cv2.dilate(thresh, kernel, iterations=1)

# --- STEP 7: OCR (Digits only) ---
custom_config = r'--oem 3 --psm 7 -c tessedit_char_whitelist=0123456789.'
ocr_text = pytesseract.image_to_string(thresh, config=custom_config).strip()

print("🔹 Raw OCR Output:", repr(ocr_text))

# --- STEP 8: Extract numeric digits using regex ---
numbers = re.findall(r"\d+\.\d+|\d+", ocr_text)

if numbers:
    digits_cleaned = [float(n) if '.' in n else int(n) for n in numbers]
    print("✅ Extracted Digits:", digits_cleaned)
else:
    print("⚠️ No digits detected.")

# --- STEP 9: Display Results for Debugging ---
cv2.imshow("Original Grayscale", gray)
cv2.imshow("Cropped ROI", roi)
cv2.imshow("Processed Threshold", thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()



🔹 Raw OCR Output: ''
⚠️ No digits detected.


In [24]:
import cv2
import pytesseract
import re

# Optional: path to tesseract.exe on Windows
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# --- Step 1: read and crop display area manually (adjust ROI coordinates) ---
img = cv2.imread(r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg")
if img is None:
    raise SystemExit("Image not found! Check your file path.")
# If display is always in same place, crop that region
x, y, w, h = 50, 200, 100, 300      # change to match your scale
crop = img[y:y+h, x:x+w]

# --- Step 2: preprocessing for cleaner OCR ---
gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (3,3), 0)
thresh = cv2.adaptiveThreshold(
    blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 11, 2
)

# --- Step 3: OCR + extract only digits or decimals ---
text = pytesseract.image_to_string(thresh, config="--psm 7 digits")
weight = re.findall(r"\d+\.?\d*", text)
print("Detected weight:", weight[0] if weight else "None")

cv2.imshow("Cropped Display", crop)
cv2.imshow("Processed", thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


Detected weight: None


In [29]:
import cv2
import pytesseract
import re
import easyocr  # add EasyOCR

# Optional: path to tesseract.exe on Windows
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# --- Step 1: read and crop display area manually (adjust ROI coordinates) ---
img = cv2.imread(r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg")
if img is None:
    raise SystemExit("Image not found! Check your file path.")
# If display is always in same place, crop that region
x, y, w, h = 50, 200, 100, 300      # change to match your scale
crop = img[y:y+h, x:x+w]

# --- Step 2: preprocessing for cleaner OCR ---
gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (3,3), 0)
thresh = cv2.adaptiveThreshold(
    blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 11, 2
)

# --- Step 3: OCR using Tesseract ---
text = pytesseract.image_to_string(thresh, config="--psm 7 digits")
weight = re.findall(r"\d+\.?\d*", text)
print("Tesseract Detected weight:", weight[0] if weight else "None")

# --- Step 4: OCR using EasyOCR ---
reader = easyocr.Reader(['en'])  # initialize English reader
result = reader.readtext(thresh, detail=0, allowlist='0123456789.')  # only digits
easyocr_weight = ''.join(result)
print("EasyOCR Detected weight:", easyocr_weight if easyocr_weight else "None")

cv2.imshow("Cropped Display", crop)
cv2.imshow("Processed", thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Tesseract Detected weight: None
EasyOCR Detected weight: None


In [30]:
import cv2
import pytesseract
import re
import numpy as np

# Path to Tesseract OCR
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# --- Step 1: Load image ---
img_path = r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg"
img = cv2.imread(img_path)
if img is None:
    raise SystemExit("❌ Image not found! Check your file path.")

# --- Step 2: Manual crop for display ---
# Adjust coordinates to fit your scale display
x, y, w, h = 50, 200, 100, 300
crop = img[y:y+h, x:x+w]

# Optional: resize crop for better visibility
scale_factor = 2
crop_resized = cv2.resize(crop, (w*scale_factor, h*scale_factor), interpolation=cv2.INTER_LINEAR)

# --- Step 3: Preprocess cropped image ---
gray = cv2.cvtColor(crop_resized, cv2.COLOR_BGR2GRAY)
# Enhance contrast
gray = cv2.convertScaleAbs(gray, alpha=2.0, beta=20)
# Blur to reduce noise
blur = cv2.GaussianBlur(gray, (3,3), 0)
# Thresholding (invert so digits are white)
thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                               cv2.THRESH_BINARY_INV, 11, 2)
# Morphological close to strengthen digits
kernel = np.ones((2,2), np.uint8)
thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

# --- Step 4: OCR ---
custom_config = r'--oem 3 --psm 7 -c tessedit_char_whitelist=0123456789.'
text = pytesseract.image_to_string(thresh, config=custom_config)

# --- Step 5: Extract numeric value ---
weight = re.findall(r"\d+\.\d+|\d+", text)
detected_weight = weight[0] if weight else "None"
print("✅ Detected Weight:", detected_weight)

# --- Step 6: Display images ---
cv2.imshow("Original Crop", crop_resized)
cv2.imshow("Processed for OCR", thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


✅ Detected Weight: None


In [23]:
import cv2
import numpy as np
import pytesseract
import re
from pytesseract import Output

# ---------- CONFIG ----------
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
img_path = r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg"
# ----------------------------

# Load image
img = cv2.imread(img_path)
if img is None:
    raise SystemExit("❌ Image not found! Check your file path.")

h_img, w_img = img.shape[:2]

# --- Step 1: Preprocessing ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray = cv2.convertScaleAbs(gray, alpha=1.6, beta=15)
blur = cv2.GaussianBlur(gray, (3, 3), 0)

# Adaptive threshold (invert for white digits on black)
th = cv2.adaptiveThreshold(
    blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2
)

# Morphological closing (joins nearby digit regions)
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
th_morph = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=1)

# --- Step 2: Detect digit regions using Tesseract boxes ---
tess_cfg = "--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789."
data = pytesseract.image_to_data(th_morph, config=tess_cfg, output_type=Output.DICT)

boxes = []
n = len(data['text'])
for i in range(n):
    txt = str(data['text'][i]).strip()
    conf = data['conf'][i]

    # Handle confidence safely (can be int or string)
    if isinstance(conf, str):
        try:
            conf = float(conf)
        except ValueError:
            conf = -1

    # Keep only valid digits with good confidence
    if txt != "" and re.search(r"\d", txt) and conf > 10:
        x, y, w, h = data['left'][i], data['top'][i], data['width'][i], data['height'][i]
        boxes.append((x, y, w, h))

# Merge found boxes into one ROI
def merge_boxes(box_list, pad=8):
    xs = [b[0] for b in box_list]
    ys = [b[1] for b in box_list]
    ws = [b[2] for b in box_list]
    hs = [b[3] for b in box_list]
    x1 = max(min(xs) - pad, 0)
    y1 = max(min(ys) - pad, 0)
    x2 = min(max([xs[i] + ws[i] for i in range(len(xs))]) + pad, w_img)
    y2 = min(max([ys[i] + hs[i] for i in range(len(ys))]) + pad, h_img)
    return x1, y1, x2, y2


roi = None
roi_source = None
if boxes:
    x1, y1, x2, y2 = merge_boxes(boxes, pad=6)
    roi = gray[y1:y2, x1:x2]
    roi_source = "tesseract_boxes"
else:
    # fallback: take middle band
    y1 = int(h_img * 0.3)
    y2 = int(h_img * 0.6)
    x1 = int(w_img * 0.2)
    x2 = int(w_img * 0.8)
    roi = gray[y1:y2, x1:x2]
    roi_source = "fallback_central"

# --- Step 3: Prepare ROI for OCR ---
if roi is None or roi.size == 0:
    raise SystemExit("⚠️ No ROI found. Try adjusting preprocessing or input image.")

# Enlarge for better OCR
h_r, w_r = roi.shape[:2]
if w_r < 200:
    scale = int(200 / w_r) + 1
    roi = cv2.resize(roi, (w_r * scale, h_r * scale), interpolation=cv2.INTER_LINEAR)

# Clean + binarize
roi_blur = cv2.GaussianBlur(roi, (3,3), 0)
_, roi_th = cv2.threshold(roi_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Invert if needed (ensure dark digits on white)
if np.mean(roi_th) < 127:
    roi_th = cv2.bitwise_not(roi_th)

# Slight morphology to thicken digits
kernel = np.ones((2,2), np.uint8)
roi_th = cv2.morphologyEx(roi_th, cv2.MORPH_CLOSE, kernel)

# --- Step 4: OCR only on the ROI ---
ocr_cfg = "--oem 3 --psm 7 -c tessedit_char_whitelist=0123456789."
ocr_text = pytesseract.image_to_string(roi_th, config=ocr_cfg).strip()
numbers = re.findall(r"\d+\.\d+|\d+", ocr_text)

# --- Step 5: Results ---
print("ROI source:", roi_source)
print("Raw OCR text:", repr(ocr_text))
print("Extracted numbers:", numbers)

# --- Visualization ---
vis = img.copy()
cv2.rectangle(vis, (x1, y1), (x2, y2), (0,255,0), 2)
cv2.imshow("Original with ROI", vis)
cv2.imshow("ROI (grayscale)", roi)
cv2.imshow("ROI (binarized for OCR)", roi_th)
cv2.waitKey(0)
cv2.destroyAllWindows()


ROI source: fallback_central
Raw OCR text: ''
Extracted numbers: []


In [10]:
import cv2
import numpy as np
import pytesseract
import re

# Path to tesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# --- Read image ---
img_path = r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg"
img = cv2.imread(img_path)
if img is None:
    raise SystemExit("❌ Image not found!")

# --- Convert to grayscale ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray = cv2.convertScaleAbs(gray, alpha=1, beta=20)

# --- Crop ROI ---
h, w = gray.shape
roi = gray[int(h * 0.3):int(h * 0.5), int(w * 0.1):int(w * 0.5)]

# --- Preprocess (blur + Otsu + morphology) ---
blur = cv2.GaussianBlur(roi, (3, 3), 0)
_, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
kernel = np.ones((2, 2), np.uint8)
thresh = cv2.dilate(thresh, kernel, iterations=1)
thresh = cv2.erode(thresh, kernel, iterations=1)

# --- OCR ---
custom_config = r'--oem 3 --psm 8 -c tessedit_char_whitelist=0123456789.'
ocr_text = pytesseract.image_to_string(thresh, config=custom_config).strip()

# --- Extract digits ---
numbers = re.findall(r"\d+\.\d+|\d+", ocr_text)
digits_cleaned = [float(n) if '.' in n else int(n) for n in numbers] if numbers else []

print("🔹 Tesseract Raw OCR Output:", repr(ocr_text))
print("✅ Extracted Digits:", digits_cleaned if digits_cleaned else "⚠️ None detected")

# --- Display for debugging ---
cv2.imshow("Cropped ROI", roi)
cv2.imshow("Processed Threshold", thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


🔹 Tesseract Raw OCR Output: ''
✅ Extracted Digits: ⚠️ None detected


In [16]:
import cv2
import pytesseract
import re
import numpy as np

# --- Step 1: Tesseract path ---
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# --- Step 2: Load image ---
img_path = r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg"
img = cv2.imread(img_path)
if img is None:
    raise SystemExit("❌ Image not found! Check your path.")

# Resize for consistency
img = cv2.resize(img, (800, 600))

# --- Step 3: Convert to grayscale ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# --- Step 4: Manual crop for weight display ---
# Adjust these coordinates until the ROI perfectly fits the digits
# You need to tweak y1:y2 and x1:x2 based on your image
y1, y2 = 220, 310    # vertical crop
x1, x2 = 150, 400    # horizontal crop
roi = gray[y1:y2, x1:x2]

# Optional: draw rectangle on original image to verify crop
cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

# --- Step 5: Preprocess ROI for OCR ---
enhanced = cv2.convertScaleAbs(roi, alpha=2.0, beta=0)  # increase contrast
blur = cv2.GaussianBlur(enhanced, (3, 3), 0)
thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY_INV, 11, 2)

# --- Step 6: OCR to extract digits ---
custom_config = r'--oem 3 --psm 7 -c tessedit_char_whitelist=0123456789.'
text = pytesseract.image_to_string(thresh, config=custom_config)

# Extract numeric value (supports decimals)
weight = re.findall(r"\d+\.\d+|\d+", text)
detected_weight = weight[0] if weight else "None"

print("✅ Detected Weight:", detected_weight)

# --- Step 7: Display images for verification ---
cv2.imshow("Original Image with ROI", img)
cv2.imshow("Cropped Weight Region", roi)
cv2.imshow("Processed for OCR", thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


✅ Detected Weight: None


In [17]:
import cv2
import pytesseract
import re
import numpy as np

# --- Step 1: Tesseract path ---
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# --- Step 2: Load image ---
img_path = r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg"
img = cv2.imread(img_path)
if img is None:
    raise SystemExit("❌ Image not found! Check your path.")

# Resize for consistency
img = cv2.resize(img, (800, 600))

# --- Step 3: Convert to grayscale ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# --- Step 4: Manual crop for weight display ---
y1, y2 = 220, 310    # vertical crop
x1, x2 = 150, 400    # horizontal crop
roi = gray[y1:y2, x1:x2]

# Optional: draw rectangle on original image to verify crop
cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

# --- Step 5: Enhance ROI ---
# 1. Resize to make digits bigger for OCR
scale_factor = 2
roi_resized = cv2.resize(roi, (roi.shape[1]*scale_factor, roi.shape[0]*scale_factor), interpolation=cv2.INTER_LINEAR)

# 2. Increase contrast
enhanced = cv2.convertScaleAbs(roi_resized, alpha=2.0, beta=20)

# 3. Blur slightly to reduce noise
blur = cv2.GaussianBlur(enhanced, (3,3), 0)

# 4. Thresholding (invert so digits are white)
thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY_INV, 11, 2)

# Optional: morphological closing to strengthen digits
kernel = np.ones((2,2), np.uint8)
thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

# --- Step 6: OCR ---
custom_config = r'--oem 3 --psm 7 -c tessedit_char_whitelist=0123456789.'
text = pytesseract.image_to_string(thresh, config=custom_config)

# Extract numeric value
weight = re.findall(r"\d+\.\d+|\d+", text)
detected_weight = weight[0] if weight else "None"

print("✅ Detected Weight:", detected_weight)

# --- Step 7: Show images ---
cv2.imshow("Original Image with ROI", img)
cv2.imshow("Cropped ROI (resized)", roi_resized)
cv2.imshow("Processed for OCR", thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


✅ Detected Weight: None


In [2]:
import cv2
import easyocr
import re

# --- Step 1: Load image ---
img_path = r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg"
img = cv2.imread(img_path)
if img is None:
    raise SystemExit("Image not found! Check your file path.")

# --- Step 2: Preprocess full image for OCR ---
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (3,3), 0)
thresh = cv2.adaptiveThreshold(
    blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 11, 2
)

# --- Step 3: OCR using EasyOCR on full image ---
reader = easyocr.Reader(['en'])
result = reader.readtext(thresh, detail=0, allowlist='0123456789.')  # only digits
easyocr_weight = ''.join(result)
print("EasyOCR Detected weight:", easyocr_weight if easyocr_weight else "None")

# --- Step 4: Display full image and processed image ---
cv2.imshow("Original Image", img)
cv2.imshow("Processed for OCR", thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


: 

In [3]:
import cv2
import easyocr
import re

# --- Step 1: Load image ---
img_path = r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg"
img = cv2.imread(img_path)
if img is None:
    raise SystemExit("Image not found! Check your file path.")

# --- Step 2: Crop Weight Display Area ---
h, w, _ = img.shape
y1, y2 = int(h * 0.26), int(h * 0.40)
x1, x2 = int(w * 0.20), int(w * 0.55)
roi = img[y1:y2, x1:x2]

# --- Step 3: Preprocess cropped ROI ---
gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (3, 3), 0)
thresh = cv2.adaptiveThreshold(
    blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2
)

# --- Step 4: OCR using EasyOCR ---
reader = easyocr.Reader(['en'])
result = reader.readtext(thresh, detail=0, allowlist='0123456789.')
print("OCR result:", result)

# --- Step 5: Extract only the weight number ---
easyocr_weight = ''.join(result)
weight_match = re.search(r'\d+\.\d+', easyocr_weight)
extracted_weight = weight_match.group(0) if weight_match else "Not found"

print("Extracted weight:", extracted_weight)

# --- Step 6: Display for debugging ---
cv2.imshow('roi',roi)
cv2.imshow('thres',thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


OCR result: []
Extracted weight: Not found


In [4]:
import cv2
import easyocr
import re

# --- Step 1: Load image ---
img_path = r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg"

img = cv2.imread(img_path)
if img is None:
    raise SystemExit("Image not found! Check your file path.")

# --- Step 2: Crop Weight Display Area ---
h, w, _ = img.shape
y1, y2 = int(h * 0.30), int(h * 0.54)
x1, x2 = int(w * 0.1), int(w * 0.45)
roi = img[y1:y2, x1:x2]

# --- Step 3: Preprocess cropped ROI ---
gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (3, 3), 0)
thresh = cv2.adaptiveThreshold(
    blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2
)

# --- Step 4: OCR using EasyOCR ---
reader = easyocr.Reader(['en'])
result = reader.readtext(thresh, detail=0, allowlist='0123456789.')
print("OCR result:", result)

# --- Step 5: Extract only the weight number ---
easyocr_weight = ''.join(result)
weight_match = re.search(r'\d+\.\d+', easyocr_weight)
extracted_weight = weight_match.group(0) if weight_match else "Not found"

print("Extracted weight:", extracted_weight)

# --- Step 6: Display for debugging ---
cv2.imshow('roi', roi)
cv2.imshow('thresh', thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


: 

In [3]:
import cv2
import os
import numpy as np

# --- Step 1: Load your image ---
img_path = r"C:\Users\World\Desktop\WhatsApp Image 2025-10-11 at 10.33.25_5c8c3d5d.jpg"

img = cv2.imread(img_path)
if img is None:
    raise SystemExit("❌ Image not found")

# --- Step 2: Crop only the display region ---
h, w, _ = img.shape
y1, y2 = int(h * 0.3), int(h * 0.55)
x1, x2 = int(w * 0.1), int(w * 0.5)
roi = img[y1:y2, x1:x2]

# --- Step 3: Convert to grayscale and threshold ---
gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
gray = cv2.GaussianBlur(gray, (3, 3), 0)
gray = cv2.bitwise_not(gray)
_, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# --- Step 4: Find contours (potential digits) ---
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

os.makedirs("digit_dataset/raw_digits", exist_ok=True)
i = 0
for cnt in sorted(contours, key=lambda c: cv2.boundingRect(c)[0]):
    x, y, w, h = cv2.boundingRect(cnt)
    if h > 20 and w > 10:  # filter out noise
        digit = thresh[y:y+h, x:x+w]
        digit = cv2.resize(digit, (64, 64))
        cv2.imwrite(f"digit_dataset/raw_digits/digit_{i}.png", digit)
        i += 1

print(f"✅ Cropped {i} digit regions saved in 'digit_dataset/raw_digits/'")


✅ Cropped 3 digit regions saved in 'digit_dataset/raw_digits/'


In [8]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image
import numpy as np
import os

augmenter = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.6, 1.4],
    zoom_range=0.2,
)

for digit_folder in os.listdir(r"C:\Users\World\digit_dataset"):
    input_path = fr"C:\Users\World\digit_dataset\{digit_folder}"
    output_path = fr"C:\Users\World\digit_dataset_augmented\{digit_folder}"
    os.makedirs(output_path, exist_ok=True)

    for img_name in os.listdir(input_path):
        img = Image.open(os.path.join(input_path, img_name)).convert("L")
        img_array = np.expand_dims(np.array(img), axis=(0, -1))  # shape (1, h, w, 1)
        i = 0
        for batch in augmenter.flow(img_array, batch_size=1, save_to_dir=output_path,
                                   save_prefix=digit_folder, save_format='png'):
            i += 1
            if i > 50:  # generate 50 variations per image
                break
